# 03. 전용 컨테이너와 준비 검사 / 45분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

준비 검사는 설치가 아니라 실행 조건의 확인입니다. 4일차와 다른 VLA 전용 이미지를 사용합니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_lab03_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![전용 컨테이너와 준비 검사의 입력·처리·출력](../교재그림/guided-vla/day5-03.svg)

노트북은 sys.executable과 ROOT를 확인한 후 같은 Python으로 preflight.py를 실행합니다. 실패 위치를 읽으면 커널 문제인지 Data 누락인지 구분할 수 있습니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![5일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day5-system.svg)

브라우저는 실행을 요청하고 결과를 표시합니다. 커널이 호출한 별도 runner가 물리 환경과 정책을 연결합니다. 01·02의 합성 셀은 구조 설명용이고 실제 환경 실행은 04, 모델 실행은 06부터입니다. 아래 단원별 도식에서 이번에 실제로 실행할 범위를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 호스트·컨테이너·커널

- 이미지는 실행 환경의 준비본, 컨테이너는 실행 인스턴스입니다.
- bind mount는 호스트 자료를 /workspace로 연결합니다.
- 노트북 커널의 Python과 터미널 Python을 구분합니다.
- 포트가 달라도 GPU는 같은 물리 장치를 공유할 수 있습니다.

컨테이너는 강의에 맞춘 Python과 라이브러리 환경을 실행합니다. 4일차 이미지와 VLA 전용 이미지는 필요한 패키지와 GUI 구성이 다르므로 같은 브라우저에서 열리는 Jupyter라는 이유로 교환해서 사용할 수 없습니다. sys.executable을 출력하면 현재 셀이 어느 Python에서 실행되는지 알 수 있습니다.

호스트의 개인 강의자료 폴더를 컨테이너의 /workspace에 bind mount합니다. 따라서 노트북과 Data의 파일 변경은 연결된 호스트 폴더에도 반영됩니다. 컨테이너 내부의 아무 경로에나 저장한 파일이 모두 영구 보존되는 것은 아닙니다. 런처는 종료 시 제거되는 컨테이너를 사용하므로 결과를 정해진 Data 아래에 두는 이유가 있습니다.

컨테이너 이름과 Jupyter/noVNC 포트는 접속과 프로세스 구분을 위한 값입니다. 개인 컨테이너가 있다고 GPU 메모리와 연산 자원이 물리적으로 분리되는 것은 아닙니다. 같은 Jetson의 정책 추론을 순서대로 실행해 부하 조건을 관리합니다.


### 2. 실행 아키텍처: 접속 경로와 계산 경로

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day5-03.svg)

- 호스트 run.sh → VLA 컨테이너 + /workspace 마운트
- 브라우저 → Jupyter → Python 커널 → 실행 스크립트
- 별도 noVNC → 가상 데스크톱 → MuJoCo 관찰 창
- CUDA·모델·캐시는 계산 경로에서 사용됩니다.

호스트 런처는 Docker 컨테이너를 만들고 workspace, 사용자, NVIDIA runtime, 서비스 포트, GUI 환경을 설정합니다. 호스트에서 실행할 명령을 Jupyter 내부 터미널에서 다시 실행하려 하면 Docker 접근이나 경로 의미가 달라질 수 있습니다. 문서의 실행 위치 표기를 먼저 읽습니다.

원격 접속에서는 SSH 터널이 브라우저의 localhost 포트를 Jetson의 서비스 포트로 연결합니다. 노트북 계산이 사용자 PC로 옮겨지는 것은 아닙니다. Jupyter는 셀 실행과 파일 접근을, noVNC는 가상 데스크톱의 GUI를 보여 줍니다. 한쪽 브라우저 탭이 닫혀도 다른 쪽 프로세스가 자동 종료되는 구조가 아닙니다.

런처는 host network를 사용하므로 실제 호스트 포트 충돌을 검사합니다. 기본 noVNC 포트는 Jupyter+100, 내부 VNC 포트는 +200이며 서로 달라야 합니다. 로컬·원격 선택은 접속 방법을 바꾸고 정책의 입출력 계약을 바꾸지 않습니다.


### 3. 코드 추적: preflight가 확인하는 범위

- check_model: manifest·revision·파일 크기 확인
- --hash 선택 시 큰 파일 SHA256까지 확인
- 캐시 디렉터리·LIBERO config.yaml·CUDA 확인
- 패키지 버전·장치 출력; 다운로드와 rollout은 하지 않음

preflight.py는 course_runtime을 import하면서 offline 환경과 캐시 경로를 설정합니다. check_model은 준비된 manifest의 저장소와 revision을 확인하고 파일 존재·크기를 검사합니다. --hash는 내용 해시 검사까지 수행하므로 큰 모델 파일을 읽는 시간이 추가됩니다. 단순 존재 검사와 내용 무결성 검사는 다른 수준입니다.

그 뒤 HF_HOME과 LIBERO_CONFIG_PATH, config.yaml, CUDA 사용 가능 여부를 확인하고 장치와 패키지 버전을 출력합니다. 이 검사에 통과해도 모든 과제 자산과 모델 계산이 끝까지 동작했다는 뜻은 아닙니다. 실제 캐시 완전성은 inspect·infer·rollout으로 단계적으로 확인합니다.

노트북은 subprocess.run에 sys.executable을 전달합니다. 커널과 다른 python을 PATH에서 우연히 찾는 문제를 줄이기 위해서입니다. check=True는 자식 프로세스의 실패를 노트북 오류로 드러냅니다. 오류를 무시하고 다음 셀로 진행하면 원래 준비 문제 대신 나중의 파일 없음만 보일 수 있습니다.


### 4. 실패 위치별 확인과 수정

- import 오류: 커널·이미지·Python 경로 확인
- 모델/캐시 누락: 개인 자료와 Data 마운트 확인
- CUDA 오류: 전용 컨테이너·NVIDIA runtime 확인
- 접속 오류: 실행 프로세스·포트·SSH 터널 확인

ModuleNotFoundError라면 우선 커널과 이미지가 맞는지 확인합니다. 모델 파일 누락이라면 현재 ROOT와 Data 경로가 개인 자료 폴더를 가리키는지 확인합니다. 임의 pip install이나 온라인 다운로드로 증상만 지우면 강의의 고정 실행 환경과 달라져 다른 문제를 만들 수 있습니다.

noVNC 접속 실패와 CUDA 추론 실패는 다른 경로의 문제입니다. 브라우저 GUI가 보인다고 GPU 정책이 준비되었다는 뜻도, GPU가 인식된다고 SSH 터널이 올바르다는 뜻도 아닙니다. preflight 출력과 서비스 접속을 나눠 읽으면 필요한 확인 범위가 줄어듭니다.

직접 런처를 수정할 때는 개인 workspace, 이름, 세 포트와 DISPLAY의 대응을 유지합니다. 모델 경로나 COURSE_DATA를 바꿨다면 후속 노트북이 읽는 결과 위치도 같은 자료를 가리키는지 확인해야 합니다. 이번 활동은 준비된 환경을 진단하는 실행 실습이며 설치 작업을 학생에게 맡기지 않습니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [scripts/preflight.py](scripts/preflight.py) |
| 바꾸어 볼 조건 | `호스트 런처의 개인 경로·이름·포트` |
| 예상되는 차이 | 검사 출력의 Python·모델 경로를 실제 개인 컨테이너와 대조합니다. preflight는 새 MuJoCo 창을 열지 않습니다. |
| 직접 구현·수정할 때 | 학생은 pip install이나 모델 다운로드로 환경을 바꾸지 않습니다. 호스트 런처와 컨테이너 명령을 구분하고 공유 GPU에서 정책은 순서대로 실행합니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


### 실제 코드 읽기 — 입력·처리·출력 연결

노트북은 sys.executable과 ROOT를 확인한 후 같은 Python으로 preflight.py를 실행합니다. 실패 위치를 읽으면 커널 문제인지 Data 누락인지 구분할 수 있습니다.


In [ ]:
from pathlib import Path
from IPython.display import Code, display
import ast
source_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'10.VLA_v2/scripts').is_dir())
source_path = source_root/'10.VLA_v2'/'scripts/preflight.py'
text = source_path.read_text()
# 함수 위치를 AST로 찾아 해당 함수 전체를 표시합니다. 파일을 import하거나 모델을 로드하지 않습니다.
functions = [node for node in ast.parse(text).body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
if not functions:
    display(Code(text, language='python'))
for node in functions:
    if node.name in ['main']:
        print(source_path, '함수:', node.name, '줄:', node.lineno)
        display(Code('\n'.join(text.splitlines()[node.lineno-1:node.end_lineno]), language='python'))


## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

03에서는 준비 검사만 수행하므로 MuJoCo 창이 아직 없습니다. noVNC 바탕 화면에 접속할 수 있는지 확인하고 04에서 초기 환경을 엽니다. 컨테이너 생성은 호스트에서 이미 완료한 단계입니다.


## 1. 지금 어디에서 확인하는 건가요?

<!-- procedure:lesson-39-f0d92c:start -->
![열린 컨테이너에서 준비 검사](../교재그림/절차/lesson-39.svg)

03은 이미 열린 컨테이너의 점검입니다. 파일·GPU 검사 통과와 실제 렌더링 성공은 다른 단계입니다.
<!-- procedure:lesson-39-f0d92c:end -->


이 노트북은 **이미 열린 5일차 VLA 컨테이너의 JupyterLab** 에서 읽고 실행합니다. 컨테이너를 새로 만드는 명령은 여기서 실행하지 않습니다. 접속·복구가 필요하면 [5일차 실행 안내](README.md#환경과-실행)를 사용합니다.

| 구성 | 제공하는 것 | 이번에 확인할 내용 |
|---|---|---|
| 전용 이미지·컨테이너 | Python과 VLA 실습 라이브러리 | 준비된 Python 환경을 사용하는가 |
| Jetson GPU | 정책 추론에 필요한 CUDA 계산 장치 | 현재 Python에서 GPU를 사용할 수 있는가 |
| `/workspace` | 호스트에서 연결한 교재 폴더 | 실제 교재와 점검 스크립트가 보이는가 |
| `/workspace/Data` | 배포 모델·캐시·환경 자산, 이후 실습 결과 | 필요한 자료가 연결되어 있는가 |

컨테이너는 실행 환경이고, 모델·교재·결과 파일은 연결한 호스트 폴더에 있습니다. 컨테이너 안의 `/workspace`라는 이름이 같아도 **연결한 호스트 개인 폴더가 다르면 내용은 다를 수 있습니다.**

5일차 가상 카메라는 EGL 방식으로 장면을 그립니다. 이 점검에는 USB 카메라나 공용 Ollama가 필요하지 않습니다. 실제 영상 생성은 다음 04에서 확인합니다.


## 2. 첫 번째 점검 — Python과 자료 위치

아래 셀은 경로를 찾아 **현재 값만 출력** 합니다. 결과 폴더를 만들거나 모델을 실행하지 않습니다.

- **Python:** 보통 `/opt/course/bin/python` 또는 같은 환경의 `python3` 경로가 나옵니다.
- **교재:** 컨테이너에서는 `/workspace`가 기준입니다.
- **Data·모델:** 다음 점검에서 읽을 실제 자료 위치입니다.
- **이후 결과 저장 기준:** 아직 결과가 생겼다는 뜻이 아니라, 다음 실습에서 사용할 위치입니다.

Python 경로가 다른 환경이라면 JupyterLab의 커널 선택에서 강사가 준비한 VLA용 Python 커널을 선택하고 이 셀부터 다시 실행하세요.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
from pathlib import Path
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / '10.VLA_v2/scripts/preflight.py').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('교재 위치를 찾지 못했습니다. /workspace/10.VLA_v2의 노트북을 열었는지 확인하세요.')

SCRIPTS = ROOT / '10.VLA_v2/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, MODEL

print('Python:', sys.executable)
print('교재:', ROOT)
print('Data:', DATA)
print('모델:', MODEL)
print('이후 결과 저장 기준:', DATA / 'logs/day5')


## 3. 두 번째 점검 — 실행 전 검사(preflight)

**preflight** 는 본 실행 전에 준비 상태를 살펴보는 검사입니다. 아래 셀은 제공된 [점검 스크립트](scripts/preflight.py)를 **현재 노트북과 같은 Python** 으로 실행합니다. 새 패키지나 모델을 다운로드하지 않습니다.

| 검사하는 것 | 확인하는 이유 |
|---|---|
| 모델 manifest의 저장소·revision과 파일 크기 | 수업용 체크포인트가 지정된 버전이고 필요한 파일이 있는지 확인 |
| Hugging Face·LIBERO 캐시 폴더와 `config.yaml` | 배포 자료의 기본 경로와 설정이 준비되었는지 확인 |
| CUDA 사용 가능 여부 | 현재 환경에서 NVIDIA GPU를 사용할 수 있는지 확인 |
| 주요 패키지의 설치 버전 조회 | 어떤 실행 라이브러리 버전이 설치되어 있는지 확인 |

**여기서 하지 않는 검사도 있습니다.** 실제 가상 영상 생성, 정책 추론, 집기 성공, 전체 캐시 파일의 완전성은 확인하지 않습니다. 기본 실행은 모델 파일의 SHA-256 해시를 전부 계산하지도 않습니다. 라이브러리 버전을 출력하는 것이 모든 조합의 호환성을 증명하는 것은 아닙니다.

아래 셀이 오류 없이 끝나면 기본 준비 검사를 통과한 것입니다. 출력 없이 오래 기다리거나 오류가 나면 반복 실행하지 말고 아래 오류 해석을 먼저 읽으세요.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
import subprocess

# 현재 커널과 같은 Python의 별도 프로세스를 실행합니다. check=True의 실패는 다음 셀로 숨기지 않습니다.
subprocess.run([sys.executable, str(SCRIPTS / 'preflight.py')], check=True)
print('기본 준비 검사 통과 — 출력 항목을 확인한 뒤 04 환경 관찰로 진행하세요.')


## 4. 출력은 이렇게 읽습니다

성공하면 `revision`, `device`, `packages`, `offline`, `note`가 포함된 결과가 표시됩니다. 값 자체를 외우거나 별도 사전에 복사할 필요는 없습니다. **각 값이 무엇을 확인해 주는지 설명할 수 있으면 됩니다.**

| 출력 항목 | 의미 | 읽는 방법 |
|---|---|---|
| `revision` | 준비된 모델 스냅샷의 식별자 | 현재 교재의 지정값 `31d453f7edd78c839a8bbc39744a292686daf0de`와 일치해야 함. 스크립트가 manifest와 대조 |
| `device` | 현재 Python에서 사용하는 CUDA 장치 이름 | 실제 장비가 보고한 이름을 읽음. 문구를 특정 GPU 이름으로 바꾸어 적지 않음 |
| `packages` | torch·lerobot·mujoco·hf-libero·robosuite의 설치 버전 | 오류 대응 시 현재 조합을 확인할 근거. 임의로 업데이트하라는 안내가 아님 |
| `offline` | Hugging Face 오프라인 설정 값 | 문자열 `"1"`이면 해당 오프라인 모드가 설정됨. 교재 코드가 이 값을 설정함 |
| `note` | 이 점검으로 확인하지 못한 범위 | 실제 환경 관찰·추론·반복 실행에서 추가 확인해야 함 |

**offline=1은 네트워크가 물리적으로 차단됐다는 뜻이 아닙니다.** 모든 자료가 빠짐없이 존재하거나 모든 프로그램의 인터넷 접근이 막혔다는 증거도 아닙니다.

`CompletedProcess(..., returncode=0)` 같은 문구가 추가로 보이면 점검 프로세스가 정상 종료됐다는 의미입니다. **점검 통과와 VLA의 작업 성공은 다릅니다.** 아직 로봇을 움직이지 않았습니다.


## 5. 오류가 나면 어디를 확인할까요?

아래는 오류에 따라 먼저 확인할 위치입니다. 점검을 통과시키려고 오류를 숨기거나 출력 값을 정답처럼 바꾸지 않습니다.

| 보이는 오류·현상 | 먼저 확인할 것 | 다시 확인하는 순서 |
|---|---|---|
| 교재 위치를 찾지 못함 | `/workspace/10.VLA_v2`의 노트북인지, 교재 폴더가 연결됐는지 | 교재 위치·마운트 확인 후 첫 셀부터 |
| `ModuleNotFoundError` 또는 패키지 버전 조회 오류 | VLA 전용 커널·이미지를 사용하는지 | 커널 확인 → 해결되지 않으면 강사에게 환경 확인 요청 |
| 모델·캐시·config 파일 누락 | 첫 셀의 Data·모델 경로와 연결한 호스트 개인 폴더 | 배포본·마운트 확인 후 점검 셀 재실행 |
| revision 불일치·모델 파일 불완전 | 지정된 모델 묶음이 연결됐는지 | 강사 배포본 확인. manifest 내용만 수정해 우회하지 않음 |
| NVIDIA runtime/CUDA 오류 | 전용 컨테이너에서 GPU가 연결됐는지 | 강사와 런처·GPU 접근 상태 확인 |
| 마지막에 `CalledProcessError` | 그 위에 출력된 구체적인 오류 | 마지막 줄은 점검 실패 요약이므로 위 원인부터 읽기 |

수강생이 임의로 패키지를 설치하거나 모델을 다운로드하는 단계가 아닙니다. 필요한 환경 복구는 [실행·복구 안내](README.md#복구)를 따릅니다. 커널을 재시작했다면 첫 셀부터 다시 실행합니다.


## 6. 결과 파일은 어디에 남나요?

**이 노트북은 점검 결과를 화면에 보여 주며, 별도의 실험 결과 파일을 만들지 않습니다.** 첫 셀에 나온 `Data/logs/day5`는 이후 실습 결과의 저장 기준입니다.

04부터는 `SESSION`이라는 이름으로 실험 회차를 정합니다. 기본 회차의 결과는 `Data/logs/day5/v2-session-01/<실행이름>/`에 저장됩니다. 같은 비교를 이어갈 때 04–08의 회차를 맞추고, 새 비교를 시작할 때 함께 변경합니다.

연결한 호스트 폴더의 파일은 컨테이너를 다시 만들어도 유지할 수 있습니다. 반면 커널 변수와 실행 중 프로세스는 복구되지 않습니다. **저장한 결과 파일과 메모리의 변수를 구분하세요.**


## 7. 다음 실습으로 넘어갈 조건

아래 세 가지를 확인하면 [04 환경 관찰](04_environment.ipynb)로 진행합니다.

1. 현재 Python과 교재·Data 위치를 확인했습니다.
2. preflight가 오류 없이 끝났고, GPU·모델 revision·offline 출력의 의미를 이해했습니다.
3. 이번에 확인한 것은 **기본 준비 상태** 이며, 가상 영상 생성과 정책 추론·작업 성공은 아직 확인하지 않았다는 것을 설명할 수 있습니다.

**이해 확인:** “점검은 통과했는데 다음 단원에서 가상 카메라가 검게 나오면 모순인가요?”

<details>
<summary>해설 보기</summary>

모순이 아닙니다. preflight는 모델 기본 파일·캐시 경로·CUDA·설치 버전을 확인하지만, 실제 EGL 렌더링으로 장면을 만들지는 않습니다. 다음 04에서 실제 환경과 영상을 확인해야 합니다. 같은 이유로 준비 점검 통과만으로 추론 속도나 작업 성공을 보장하지 않습니다.

</details>


## 결과 해설과 다음 단계

준비 검사는 설치가 아니라 실행 조건의 확인입니다. 4일차와 다른 VLA 전용 이미지를 사용합니다.

학생은 pip install이나 모델 다운로드로 환경을 바꾸지 않습니다. 호스트 런처와 컨테이너 명령을 구분하고 공유 GPU에서 정책은 순서대로 실행합니다.

준비 검사가 끝나면 모델 없이 환경을 reset하여 초기 영상부터 확인합니다.

출력 표·그래프·영상과 강사 설명을 함께 확인합니다. 별도 관찰 답안을 작성하지 않습니다.
